# AutoLyrics — Whisper Fine-Tuning

> Fine-tune OpenAI Whisper-small on the `gmenon/slt-lyrics-audio` dataset using LoRA (PEFT).

---

In [ ]:
# ================================
# 1. Install required packages (run once)
# ================================
# !pip install -U torchao jiwer transformers datasets peft librosa openai-whisper accelerate

In [ ]:
# ================================
# 2. Imports and constants
# ================================
import os
import re
import json
import copy
import gc
import random
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
import librosa
import whisper
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from datasets import load_dataset
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from jiwer import wer, cer
from difflib import SequenceMatcher

# Constants
MODEL_NAME = "openai/whisper-small"  # Base transformer model
DATASET_NAME = "gmenon/slt-lyrics-audio"
OUTPUT_DIR = "./whisper-lora-autolyrics"
LANGUAGE = "en"
TASK = "transcribe"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

TARGET_SR = 16000
MAX_DURATION = 30.0
MIN_DURATION = 2.0
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
BATCH_SIZE = 2
MAX_LABEL_TOKENS = 448
SEED = 42

# Training hyperparameters
LR = 5e-5
NUM_EPOCHS = 20
EVAL_EVERY = 1
N_VAL_SAMPLES = 50
BEAM_SIZE = 1                     # greedy decoding – safer for fine-tuning
PATIENCE = 5
GRAD_ACCUM_STEPS = 4
# MAX_NEW_TOKENS will be set dynamically after dataset creation

SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ================================
# 3. Load dataset from Hugging Face
# ================================
print(f"Loading dataset: {DATASET_NAME} ...")
# gmenon/slt-lyrics-audio has a single 'train' split
hf_dataset = load_dataset(DATASET_NAME, split="train", trust_remote_code=True)
print(f"Dataset loaded. Columns: {hf_dataset.column_names}")
print(f"Total examples: {len(hf_dataset)}")
# Inspect one example to confirm column names
print("Sample keys:", list(hf_dataset[0].keys()))

In [ ]:
# ================================
# 4. Helper functions
# ================================
def load_audio_array(audio_array, orig_sr):
    """Convert HF audio dict to a mono float32 waveform tensor at TARGET_SR."""
    waveform = torch.tensor(audio_array, dtype=torch.float32)
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)          # (1, T)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if orig_sr != TARGET_SR:
        waveform = T.Resample(orig_sr, TARGET_SR)(waveform)
    return waveform.squeeze(0)                    # (T,)

def load_audio(wav_path: str):
    """Load a wav/mp3 file from disk (kept for test-set reload compatibility)."""
    waveform, sr = torchaudio.load(wav_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != TARGET_SR:
        waveform = T.Resample(sr, TARGET_SR)(waveform)
    return waveform.squeeze(0)

In [ ]:
# ================================
# 5. Parse gmenon/slt-lyrics-audio
# ================================
def parse_slt_lyrics(hf_ds):
    """
    Convert gmenon/slt-lyrics-audio rows into the sample dicts the rest
    of the pipeline expects.

    Expected HF columns (adjust if the dataset columns differ):
      audio  -> dict with keys 'array' (np.ndarray) and 'sampling_rate'
      text   -> ground-truth lyric string
      singer -> singer / track identifier (used for train/val/test split)

    If the dataset uses different column names, update the key lookups below.
    """
    # Ensure audio column is decoded
    hf_ds = hf_ds.cast_column("audio", hf_ds.features["audio"])

    samples = []
    # Identify the lyrics column (try common names)
    text_col  = next((c for c in ["text", "lyrics", "transcript", "transcription", "sentence"] if c in hf_ds.column_names), None)
    singer_col = next((c for c in ["singer", "speaker_id", "speaker", "artist"] if c in hf_ds.column_names), None)

    if text_col is None:
        raise ValueError(f"Cannot find lyrics column. Available: {hf_ds.column_names}")

    for i, row in enumerate(tqdm(hf_ds, desc="Parsing slt-lyrics-audio")):
        audio_info = row["audio"]
        transcript = row[text_col].strip()
        if not transcript:
            continue

        orig_sr = audio_info["sampling_rate"]
        waveform = load_audio_array(audio_info["array"], orig_sr)
        duration = len(waveform) / TARGET_SR

        if duration < MIN_DURATION or duration > MAX_DURATION:
            continue

        # Use singer column if present, else bucket by index for splitting
        singer = str(row[singer_col]) if singer_col else f"spk_{i % 20:03d}"

        samples.append({
            "audio_array": audio_info["array"],   # kept in memory (small dataset)
            "audio_sr":    orig_sr,
            "transcript":  transcript,
            "source":      "slt_lyrics",
            "singer":      singer,
            "start_sec":   0.0,
            "end_sec":     duration,
        })

    print(f"[slt-lyrics-audio] Loaded {len(samples)} usable samples.")
    return samples

print("Parsing dataset...")
all_samples = parse_slt_lyrics(hf_dataset)
print(f"Total raw samples: {len(all_samples)}")

if len(all_samples) == 0:
    raise RuntimeError("No samples found! Check dataset columns and duration filters.")

In [ ]:
# ================================
# 6. Build train / val / test splits
# ================================
def split_dataset(all_samples):
    singers = {s["singer"] for s in all_samples}
    singer_list = list(singers)
    random.shuffle(singer_list)
    n_train = int(len(singer_list) * TRAIN_RATIO)
    n_val   = int(len(singer_list) * VAL_RATIO)
    train_singers = set(singer_list[:n_train])
    val_singers   = set(singer_list[n_train:n_train + n_val])
    test_singers  = set(singer_list[n_train + n_val:])

    train = [s for s in all_samples if s["singer"] in train_singers]
    val   = [s for s in all_samples if s["singer"] in val_singers]
    test  = [s for s in all_samples if s["singer"] in test_singers]
    return train, val, test


class LyricsDataset(Dataset):
    def __init__(self, samples, processor):
        self.samples   = samples
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        waveform = load_audio_array(s["audio_array"], s["audio_sr"])

        max_samples = int(MAX_DURATION * TARGET_SR)
        if len(waveform) < max_samples:
            waveform = torch.nn.functional.pad(waveform, (0, max_samples - len(waveform)))
        else:
            waveform = waveform[:max_samples]

        features = self.processor.feature_extractor(
            waveform.numpy(), sampling_rate=TARGET_SR, return_tensors="pt"
        ).input_features[0]

        labels = self.processor.tokenizer(
            s["transcript"], add_special_tokens=True
        ).input_ids[:MAX_LABEL_TOKENS]

        return {
            "input_features": features,
            "labels":         torch.tensor(labels, dtype=torch.long),
            "transcript":     s["transcript"],
        }


def collate_fn(batch):
    input_features = torch.stack([b["input_features"] for b in batch])
    max_len = max(len(b["labels"]) for b in batch)
    labels  = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        labels[i, :len(b["labels"])] = b["labels"]
    return {
        "input_features": input_features,
        "labels":         labels,
        "transcript":     [b["transcript"] for b in batch],
    }


train_data, val_data, test_data = split_dataset(all_samples)
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

# Load processor
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)

# Create datasets
train_dataset = LyricsDataset(train_data, processor)
val_dataset   = LyricsDataset(val_data,   processor)
test_dataset  = LyricsDataset(test_data,  processor)

# Dynamically set MAX_NEW_TOKENS
all_label_lengths = [
    len(processor.tokenizer(s["transcript"], add_special_tokens=True).input_ids)
    for s in train_data
]
MAX_NEW_TOKENS = int(np.percentile(all_label_lengths, 99)) + 10
print(f"Setting MAX_NEW_TOKENS = {MAX_NEW_TOKENS}")

In [ ]:
# ================================
# 7. Load base model and apply LoRA
# ================================
def load_base_model():
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_NAME, torch_dtype=MODEL_DTYPE
    ).to(DEVICE)
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    return model

def apply_lora(model, target_modules=None):
    if target_modules is None:
        module_names = set()
        for name, _ in model.named_modules():
            parts = name.split(".")
            if parts[-1] in ("q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"):
                module_names.add(parts[-1])
        target_modules = sorted(module_names)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        target_modules=target_modules,
    )
    model = get_peft_model(model, lora_config)
    return model

print("Loading base Whisper model...")
base_model = load_base_model()
print("Applying LoRA (encoder+decoder)...")
model = apply_lora(base_model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total params: {total/1e6:.1f}M, Trainable: {trainable/1e6:.3f}M ({100*trainable/total:.2f}%)")

In [ ]:
# ================================
# 8. Load base model and apply LoRA
# ================================
def load_base_model():
    model = WhisperForConditionalGeneration.from_pretrained(
        MODEL_NAME, torch_dtype=MODEL_DTYPE
    ).to(DEVICE)
    model.config.forced_decoder_ids = None
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []
    return model

def apply_lora(model, target_modules=None):
    if target_modules is None:
        module_names = set()
        for name, _ in model.named_modules():
            parts = name.split(".")
            if parts[-1] in ("q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"):
                module_names.add(parts[-1])
        target_modules = sorted(module_names)

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=16,                # reduced from 32 for stability
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        target_modules=target_modules,
    )
    model = get_peft_model(model, lora_config)
    return model

print("Loading base Whisper model...")
base_model = load_base_model()
print("Applying LoRA (encoder+decoder)...")
model = apply_lora(base_model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Total params: {total/1e6:.1f}M, Trainable: {trainable/1e6:.3f}M ({100*trainable/total:.2f}%)")

In [ ]:
# ================================
# 9. Evaluation function (fixed for PEFT)
# ================================
def evaluate(eval_model, dataset, label="val", n_samples=None):
    if n_samples:
        dataset = Subset(dataset, range(min(n_samples, len(dataset))))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    total_loss = 0.0
    all_preds = []
    all_refs = []

    # Unwrap to the inner WhisperForConditionalGeneration for both loss and generate().
    # Calling .eval() on the inner model disables dropout so val metrics are deterministic;
    # we restore training mode afterwards if the caller was training.
    if hasattr(eval_model, "base_model"):
        gen_model = eval_model.base_model.model
    else:
        gen_model = eval_model

    was_training = gen_model.training
    gen_model.eval()

    with torch.no_grad():
        for batch in tqdm(loader, desc=f"  eval [{label}]", leave=False):
            feats = batch["input_features"].to(DEVICE, dtype=MODEL_DTYPE)
            labels = batch["labels"].to(DEVICE)

            loss = gen_model(input_features=feats, labels=labels).loss  # FIX: use unwrapped model — PEFT wrapper conflicts with input_features kwarg
            total_loss += loss.item()

            generated_ids = gen_model.generate(
                input_features=feats,
                num_beams=BEAM_SIZE,
                max_new_tokens=MAX_NEW_TOKENS,
                language="en",
                task="transcribe",
                suppress_tokens=[],
                forced_decoder_ids=None,
            )
            preds = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            all_preds.extend(preds)
            all_refs.extend(batch["transcript"])

    if was_training:
        gen_model.train()

    avg_loss = total_loss / len(loader)
    pairs = [(r.strip(), p.strip()) for r, p in zip(all_refs, all_preds) if r.strip() and p.strip()]
    if not pairs:
        return avg_loss, 1.0, 1.0
    refs = [p[0] for p in pairs]
    hyps = [p[1] for p in pairs]
    avg_wer = wer(refs, hyps)
    avg_cer = cer(refs, hyps)
    return avg_loss, avg_wer, avg_cer

In [ ]:
# ================================
# 10. Training loop with gradient accumulation
# ================================
def train_model(peft_model, experiment_name, ckpt_dir):
    # Renamed parameter: peft_model (was train_model — shadowed the function name)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, peft_model.parameters()), lr=LR
    )
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
    num_steps = NUM_EPOCHS * len(train_loader)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*num_steps), num_training_steps=num_steps)

    peft_model.train()
    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    print(f"\n{'='*60}\nTraining: {experiment_name}")
    print(f"Epochs={NUM_EPOCHS}, LR={LR}, Batch={BATCH_SIZE}, GradAccum={GRAD_ACCUM_STEPS}")
    print(f"{'='*60}")

    # Unwrap once outside the epoch loop — avoids repeated attribute lookups and
    # sidesteps the PEFT Seq2SeqLM wrapper's conflict with Whisper's input_features kwarg.
    # Gradients still flow through LoRA layers since they live inside inner_model.
    inner_model = peft_model.base_model.model

    global_step = 0
    for epoch in range(1, NUM_EPOCHS+1):
        epoch_loss = 0.0
        optimizer.zero_grad()
        for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")):
            feats = batch["input_features"].to(DEVICE, dtype=MODEL_DTYPE)
            labels = batch["labels"].to(DEVICE)
            loss = inner_model(input_features=feats, labels=labels).loss
            loss = loss / GRAD_ACCUM_STEPS
            loss.backward()
            epoch_loss += loss.item() * GRAD_ACCUM_STEPS

            if (i+1) % GRAD_ACCUM_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
            global_step += 1

        # Flush any remaining gradients from the last partial accumulation window
        if (i+1) % GRAD_ACCUM_STEPS != 0:
            torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        avg_train_loss = epoch_loss / len(train_loader)

        if epoch % EVAL_EVERY == 0:
            val_loss, val_wer, val_cer = evaluate(peft_model, val_dataset, label="val", n_samples=N_VAL_SAMPLES)
            history.append({
                "epoch": epoch,
                "train_loss": avg_train_loss,
                "val_loss": val_loss,
                "val_wer": val_wer,
                "val_cer": val_cer
            })
            print(f"Epoch {epoch}: train_loss={avg_train_loss:.4f}, val_loss={val_loss:.4f}, WER={val_wer*100:.2f}%, CER={val_cer*100:.2f}%")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = copy.deepcopy({k: v.cpu() for k, v in peft_model.state_dict().items() if "lora" in k})
                patience_counter = 0
                print(f"  ✓ New best val_loss={val_loss:.4f}")
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print(f"Early stopping at epoch {epoch}")
                    break

    if best_state is not None:
        current = peft_model.state_dict()
        for k, v in best_state.items():
            if k in current:
                current[k] = v.to(DEVICE)
        peft_model.load_state_dict(current)
        print(f"Restored best model with val_loss={best_val_loss:.4f}")

    peft_model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    return history

In [ ]:
# ================================
# 11. Run experiments
# ================================
print("\nEXPERIMENT 1: Zero-shot baseline")
zs_loss, zs_wer, zs_cer = evaluate(model, val_dataset, label="zero-shot", n_samples=N_VAL_SAMPLES)
print(f"Zero-shot: Loss={zs_loss:.4f}, WER={zs_wer*100:.2f}%, CER={zs_cer*100:.2f}%")

print("\nEXPERIMENT 2: LoRA fine-tuning (encoder+decoder)")
history = train_model(model, "LoRA Enc+Dec", f"{SAVE_DIR}/lora_best")

final_loss, final_wer, final_cer = evaluate(model, val_dataset, label="final", n_samples=N_VAL_SAMPLES)
print(f"\nFinal validation: WER={final_wer*100:.2f}% (improvement from {zs_wer*100:.2f}%)")

results = {
    "zero_shot": {"wer": zs_wer, "cer": zs_cer, "loss": zs_loss},
    "lora_best": {"wer": final_wer, "cer": final_cer, "loss": final_loss, "history": history}
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {SAVE_DIR}/results.json")

In [ ]:
# ================================
# 12. Evaluation on test set
# ================================
print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

base_test = load_base_model()
best_model = PeftModel.from_pretrained(base_test, f"{SAVE_DIR}/lora_best").to(DEVICE)

test_loss, test_wer, test_cer = evaluate(best_model, test_dataset, label="test")
print(f"Test set: WER={test_wer*100:.2f}%, CER={test_cer*100:.2f}%, Loss={test_loss:.4f}")

improvement = (zs_wer - test_wer) / zs_wer * 100
print(f"Relative improvement over zero-shot: {improvement:.1f}%")
if improvement >= 15:
    print("✓ Target achieved: >15% WER reduction")
else:
    print("✗ Target not yet reached – consider more data or larger model")

test_results = {
    "wer": test_wer,
    "cer": test_cer,
    "loss": test_loss,
    "improvement_percent": improvement,
}
with open(f"{SAVE_DIR}/test_results.json", "w") as f:
    json.dump(test_results, f, indent=2)
print("\nAll done!")